# SCH-MGMT 661: Applications of AI Models  
**Instructor:** Indika Dissanayake  

---
### Tutorial: Object Detection with YOLO

---

### Dataset: Hard Hat Workers
### Source: Kaggle https://www.kaggle.com/datasets/andrewmvd/hard-hat-detection

The dataset contains two main folders:

1. **`hardhat_data/images/`**:  
   - Contains 5000 **image files** (`.png` format) showing construction scenes with workers.

2. **`hardhat_data/annotations/`**:  
   - Contains 5000 **annotation files** (`.xml` format, PASCAL VOC format) describing the bounding boxes for hard hats, heads, and persons inside each image. Each `.xml` file stores:

    - *Object class* (e.g., `hard_hat`)
    - *Bounding box coordinates*:  
       - `xmin`, `ymin`, `xmax`, `ymax`
    - *Image width and height (entire image and not individual object)*

#Set up Kaggle API Access and Download Data

- Go to Kaggle Account Settings and Create new API Token https://www.kaggle.com/settings
- Download your Kaggle.json and upload it to Colab
- Use Kaggle API to download datasets directly into Colab


In [ ]:
#import kagglehub
#kagglehub.login()

In [ ]:
from google.colab import files

# Upload your kaggle.json
files.upload()


In [ ]:
!pip install -q kaggle ultralytics
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d andrewmvd/hard-hat-detection
!unzip -q hard-hat-detection.zip -d hardhat_data

In [ ]:
import os

# Path where images are stored
images_path = '/content/hardhat_data/images'
annotations_path = '/content/hardhat_data/annotations'


# List all image/annotations files
image_files = os.listdir(images_path)
annotation_files = os.listdir(annotations_path)

# Print the number of images
print(f"Total images: {len(image_files)}")
print(f"Total annotations: {len(annotation_files)}")

# Check file formats
print(f"Image files sample: {image_files[:5]}")
print(f"Annotation files sample: {annotation_files[:5]}")



# Converting VOC Annotations to YOLO Format

In the original dataset:
- Images are stored in `.png` format.
- Annotations are in `.xml` files (PASCAL VOC format).

YOLO expects labels in `.txt` files, with each line describing:
- *class_id x_center y_center width height*



After running the conversion, you will have:
- Images: `/content/hardhat_data/images/`
- YOLO labels: `/content/hardhat_yolo/labels/`

### Task 1: Complete the bounding box coordinate conversion

You are given `xmin`, `xmax`, `ymin`, `ymax`, as well as `img_width` and `img_height` (image size in pixels).

✅ Your goal: Convert bounding box from (xmin, ymin, xmax, ymax) format to (x_center, y_center, width, height) format, normalized between 0 and 1.

- `x_center`: Horizontal center of the box (normalized)
- `y_center`: Vertical center of the box (normalized)
- `width`: Width of the box (normalized)
- `height`: Height of the box (normalized)

```
x_center = ((xmin + xmax) / 2) / img_width
y_center = ((ymin + ymax) / 2) / img_height
width = (xmax - xmin) / img_width          
height = (ymax - ymin) / img_height        
```

👉 Complete the code below:


In [ ]:
import os
import xml.etree.ElementTree as ET

# Class name to ID mapping
class_mapping = {
    "head": 0,
    "helmet": 1
}

def voc_to_yolo(xml_path, output_dir):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    with open(f"{output_dir}/{os.path.splitext(xml_path.split('/')[-1])[0]}.txt", "w") as f:
        for obj in root.findall("object"):
            cls = obj.find("name").text  # e.g., "hard_hat"
            class_id = class_mapping.get(cls)

            # Skip unknown classes (if any)
            if class_id is None:
                continue

            bbox = obj.find("bndbox")
            xmin = float(bbox.find("xmin").text)
            ymin = float(bbox.find("ymin").text)
            xmax = float(bbox.find("xmax").text)
            ymax = float(bbox.find("ymax").text)

            # Convert to YOLO format (normalized xywh)
            img_width = float(root.find("size/width").text)
            img_height = float(root.find("size/height").text)

            # -- complete this code
            x_center = __________  # 👈Fill in the blank
            y_center = __________  # 👈Fill in the blank
            width  = _______       # 👈Fill in the blank
            height = _______       # 👈Fill in the blank

            f.write(f"{class_id} {x_center} {y_center} {width} {height}\n")

# Process all XML files
os.makedirs("hardhat_yolo/labels", exist_ok=True)

for xml_file in os.listdir("hardhat_data/annotations"):
    voc_to_yolo(f"hardhat_data/annotations/{xml_file}", "hardhat_yolo/labels")

print("Bounding box conversion completed!")


# Train / Validation / Test Split

After converting all annotations to YOLO format, we need to organize the dataset for training.

Typically, we split into:

- **Training set**: 70% of the images  
  (Used to train the model — the model learns from this data.)
- **Validation set**: 20% of the images  
  (Used to tune the model during training — the model checks its performance after each epoch.)
- **Test set**: 10% of the images  
  (Used to evaluate final model performance — not touched during training.)

---

Each image must be paired with its corresponding YOLO label file.

---

## Steps:

- Randomly shuffle all available images.
- Split them into **train**, **validation**, and **test** subsets according to the defined ratios.
- For each image, move:
  - The image file into the appropriate `images/` folder.
  - The corresponding label file into the appropriate `labels/` folder.

---

After running the split:
- Images and labels will be organized under `/content/hardhat_dataset/`.
- Ready for training a YOLOv8 model!



### Task 2: Define dataset split ratios

You need to divide your dataset into training, validation, and test sets.

✅ Your goal: Assign values to the split ratios such that:
- 70% of the data goes to training
- 20% to validation
- 10% to testing

👉 Complete the code below:

In [ ]:
import os
import random
import shutil

# Set paths
images_dir = '/content/hardhat_data/images'   # PNG images
labels_dir = '/content/hardhat_yolo/labels'    # YOLO labels (.txt)

# Create new folders
for split in ['train', 'valid', 'test']:
    os.makedirs(f'/content/hardhat_dataset/{split}/images', exist_ok=True)
    os.makedirs(f'/content/hardhat_dataset/{split}/labels', exist_ok=True)

# Get list of all image filenames
images = os.listdir(images_dir)
images = [img for img in images if img.endswith('.png')]  # Only .png images
random.shuffle(images)

# Define split ratios -- complete this code
train_split = __  # 👈Fill in the blank
valid_split = __  # 👈Fill in the blank
test_split =  __  # 👈Fill in the blank

# Calculate split sizes
num_images = len(images)
train_end = int(train_split * num_images)
valid_end = train_end + int(valid_split * num_images)

train_images = images[:train_end]
valid_images = images[train_end:valid_end]
test_images = images[valid_end:]

# Helper function to move images + labels
def move_files(file_list, split):
    for img_file in file_list:
        # Copy image
        shutil.copy(os.path.join(images_dir, img_file), f'/content/hardhat_dataset/{split}/images/{img_file}')
        # Copy corresponding label
        label_file = img_file.replace('.png', '.txt')
        shutil.copy(os.path.join(labels_dir, label_file), f'/content/hardhat_dataset/{split}/labels/{label_file}')

# Move files
move_files(train_images, 'train')
move_files(valid_images, 'valid')
move_files(test_images, 'test')

print(f"Splitting complete! ({len(train_images)} train, {len(valid_images)} valid, {len(test_images)} test)")


# Creating `data.yaml` for YOLOv8

YOLOv8 requires a special configuration file called `data.yaml` to understand:

- Where the training and validation images are located.
- How many classes exist.
- What the class names are.

---

## Structure of `data.yaml`:

```yaml
path: /content/hardhat_dataset
train: train/images
val: valid/images
test: test/images

names:
  0: head
  1: helmet



In [ ]:
# Create the content for data.yaml
data_yaml = """
path: /content/hardhat_dataset
train: train/images
val: valid/images
test: test/images

names:
  0: head
  1: helmet
"""

# Write it to a file
with open('/content/hardhat_dataset/data.yaml', 'w') as f:
    f.write(data_yaml)

print("data.yaml file created successfully!")

In [ ]:
# Optional: you may print the object counts to see how blance your dataset is.

from collections import Counter
import os

# Paths to label folders
label_folders = [
    '/content/hardhat_dataset/train/labels',
    '/content/hardhat_dataset/valid/labels',
    '/content/hardhat_dataset/test/labels'
]

class_counts = Counter()

# Loop through each split
for folder in label_folders:
    for label_file in os.listdir(folder):
        with open(os.path.join(folder, label_file), 'r') as f:
            for line in f:
                class_id = int(line.split()[0])
                class_counts[class_id] += 1




In [ ]:
# Print final counts
print("Dataset Class Distribution:")
for class_id, count in class_counts.items():
    class_name = "head" if class_id == 0 else "helmet"
    print(f"Class {class_id} ({class_name}): {count} instances")

print(f"Total annotations: {sum(class_counts.values())}")

# Training YOLOv8n on Hard Hat Dataset

In this example, we will use the **YOLOv8n** model:
- "n" stands for **Nano** — a small, lightweight model ideal for faster training and quick experiments.

---

## Task 3: Training Setup:

- **Pretrained weights**: We start from pretrained `yolov8n.pt` weights.
- **Image size**: 640×640 pixels.
- **Batch size**: 16 images per batch.
- **Epochs**: 3 training passes over the data.
- **Project name**: `hardhat_detection_project`
- **Run name**: `yolov8n_transfer_learning`


👇 Complete the missing parameters in the train() function below:

In [ ]:
from ultralytics import YOLO

# Load YOLOv8 Nano model
model = YOLO('yolov8n.pt')  # Pretrained YOLOv8n weights

# Train with transfer learning (freeze most layers) -- fill in the missing parameters
model.train(
    data='/content/hardhat_dataset/data.yaml',  # Path to data.yaml
    epochs= __, # 👈Fill in the blank
    imgsz=640,
    batch= __, # 👈Fill in the blank
    freeze=22,  # Freeze backbone + neck, train only head
    lr0=0.001,
    cos_lr=True,   # Smooth LR schedule
    plots=True,    # Saves training graphs
    project="hardhat_detection_project",
    name="yolov8n_transfer_learning",
)

#Evaluate Your Trained Model

We evaluate our fine-tuned YOLO model on the **validation set** and examine:

- mAP@0.5–0.95: overall detection quality
- mAP@0.5: easier IoU threshold, more forgiving
- Precision: when the model predicts a helmet/head, how often is it correct?
- Recall: how many true helmets/heads did we find?

We also look at **per-class metrics** for:
- `head`
- `helmet`


In [ ]:

# Load your trained model
model = YOLO('/content/hardhat_detection_project/yolov8n_transfer_learning/weights/best.pt')

# Validate the model with validation dataset
metrics = model.val(
    data='/content/hardhat_dataset/data.yaml',  # Reuse your same data.yaml
    split = 'val' # this is the default
)

# Print main metrics
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP75: {metrics.box.map75:.4f}")
print(f"Precision (mean across all classes): {metrics.box.mp:.4f}")
print(f"Recall (mean across all classes): {metrics.box.mr:.4f}")

# Per-class metrics
print(f"\nPER-CLASS METRICS:")
for i, class_name in enumerate(['head', 'helmet']):
    print(f"Class {i} ({class_name}):")
    print(f"  Precision: {metrics.box.p[i]:.4f}")
    print(f"  Recall: {metrics.box.r[i]:.4f}")
    print(f"  mAP50: {metrics.box.ap50[i]:.4f}")
    print(f"  mAP50-95: {metrics.box.ap[i]:.4f}")

# Pedict and Compare Models (Pretrained vs Fine-Tuned)

In [ ]:
import matplotlib.pyplot as plt

# Load models
pretrained = YOLO('yolov8n.pt')
fine_tuned = YOLO('/content/hardhat_detection_project/yolov8n_transfer_learning/weights/best.pt')

# Evaluate both models
print("Model Comparison:")
p_metrics = pretrained.val(data='/content/hardhat_dataset/data.yaml', verbose=False)
f_metrics = fine_tuned.val(data='/content/hardhat_dataset/data.yaml', verbose=False)

print(f"mAP50-95: {p_metrics.box.map:.3f} → {f_metrics.box.map:.3f}")
print(f"Improvement: {f_metrics.box.map - p_metrics.box.map:+.3f}")

# Test on 3 random images
test_images = random.sample(os.listdir('/content/hardhat_dataset/test/images'), 3)

plt.figure(figsize=(15, 10))

for i, img_name in enumerate(test_images):
    img_path = f'/content/hardhat_dataset/test/images/{img_name}'

    # Get predictions WITH bounding boxes
    p_results = pretrained.predict(source=img_path, conf=0.25, save=False)
    f_results = fine_tuned.predict(source=img_path, conf=0.25, save=False)

    # Plot pretrained results
    plt.subplot(3, 2, i*2+1)
    p_plot = p_results[0].plot()  # This includes bounding boxes
    plt.imshow(p_plot)
    plt.title("Pretrained Model")
    plt.axis('off')

    # Plot fine-tuned results
    plt.subplot(3, 2, i*2+2)
    f_plot = f_results[0].plot()  # This includes bounding boxes
    plt.imshow(f_plot)
    plt.title("Fine-tuned Model")
    plt.axis('off')

plt.tight_layout()
plt.show()

### Let's Reflect on Our Findings

Now let’s take a moment to discuss our results and what they mean.  

- What does the precision-recall curve tell us about the model’s strengths and weaknesses?
- How did **mAP@0.5** and **mAP@0.5:0.95** evolve during training?
- Are the predictions reliable enough for real-world safety applications?
- What could we improve — more training, better annotations, or higher-resolution images?

Take a few minutes to summarize your observations. This reflection will help connect the technical outputs to practical decision-making and guide your next steps in model improvement.
